In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('playstore_limpio.csv')
print(f"Total de registros cargados: {len(df)}")
df.head(3)

Total de registros cargados: 9636


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up


Reto 1: La Trampa de las Unidades de Medida
El equipo de infraestructura quiere saber si las aplicaciones más pesadas tienen menos descargas. Para averiguarlo, necesitan que la columna Size sea completamente numérica. Sin embargo, si revisan la columna, encontrarán valores como "19M" (Megabytes), "201k" (Kilobytes) y un texto molesto que dice "Varies with device".

Tu misión algorítmica:
Escribe una función pura en Python que reciba un string. Si el string termina en 'M', quítale la 'M' y conviértelo a flotante (dejándolo como Megabytes). Si termina en 'k', quítale la 'k', conviértelo a flotante y divídelo entre 1024 (para pasarlo también a Megabytes). Si dice "Varies with device", devuélvelo como un valor nulo de Numpy (np.nan).
Aplica esta función a toda la columna utilizando el método .apply().
Pregunta a responder: Una vez convertida la columna a valores numéricos (Megabytes), ejecuta el método .mean(). ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?

Respuesta: El peso promedio de las aplicaciones es de aproximadamente 22.28 MB despues de aplicar la funcion

In [ ]:
def convertir_tamano(valor):
    if not isinstance(valor, str):
        return np.nan
    valor = valor.strip()
    if valor == 'Varies with device':
        return np.nan
    elif valor.endswith('M'):
        return float(valor[:-1])
    elif valor.endswith('k'):
        return float(valor[:-1]) / 1024
    return np.nan

df['Size'] = df['Size'].apply(convertir_tamano)

peso_promedio = df['Size'].mean()
print(f"Peso promedio de las apps: {peso_promedio:.2f} MB")

Peso promedio de las apps: 20.42 MB


Reto 2: El Tipo de Dato Cronológico
Ningún análisis de software está completo sin analizar el tiempo. La columna Last Updated tiene fechas escritas como texto: "January 7, 2018". Para un modelo matemático o una serie de tiempo, eso es texto inservible.

Tu misión algorítmica:
Investiga y utiliza la función pd.to_datetime() de Pandas para sobrescribir la columna Last Updated, convirtiéndola del tipo string (Object) al tipo nativo datetime64.
Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada Year_Updated extrayendo únicamente el año (df['Last Updated'].dt.year).
Pregunta a responder: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

Respuesta: El año con la mayor cantidad de aplicaciones actualizadas fue 2018

In [ ]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'])

df['Year_Updated'] = df['Last Updated'].dt.year

conteo_anios = df['Year_Updated'].value_counts()
print("Aplicaciones actualizadas por año:")
print(conteo_anios)
print(f"\nAño con más actualizaciones: {conteo_anios.idxmax()} ({conteo_anios.max()} apps)")

Aplicaciones actualizadas por año:
Year_Updated
2018    6271
2017    1785
2016     779
2015     448
2014     203
2013     108
2012      26
2011      15
2010       1
Name: count, dtype: int64

Año con más actualizaciones: 2018 (6271 apps)


Reto 3: La Decisión Arquitectónica
Al resolver el Reto 1, introdujiste intencionalmente valores NaN en las aplicaciones cuyo tamaño decía "Varies with device".

Tu misión algorítmica: Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?
Pregunta a responder: Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

Respuesta: Creo que es mejor con la mediana global en lugar de eliminar filas para no descartar informacion que puede ser considerada valiosa de otras variables ni llegar a tener algun sesgo de seleccion, tambien la mediana es insensible a la asimetria y outliers presentes en los pesos de aplicaciones.

In [ ]:
nulos_size = df['Size'].isna().sum()
porcentaje = (nulos_size / len(df)) * 100
print(f"Registros vacios en 'Size': {nulos_size} ({porcentaje:.2f}% del dataset)")

mediana_size = df['Size'].median()
df['Size'] = df['Size'].fillna(mediana_size)

print(f"Mediana global utilizada para imputar: {mediana_size:.2f} MB")
print(f"Nulos restantes en 'Size': {df['Size'].isna().sum()}")

Registros vacíos en 'Size': 0 (0.00% del dataset)
Mediana global utilizada para imputar: 12.00 MB
Nulos restantes en 'Size': 0
